In [0]:
import mlflow
import mlflow.sklearn
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from mlflow.models.signature import infer_signature

#Set experiment 

In [0]:
mlflow.set_experiment("/Users/approjects8473@gmail.com/customer_segmentation_v3")
print("MLflow experiment set up successfully")

#Load Gold tables

In [0]:
fact_sales = spark.table("workspace.gold.fact_sales").toPandas()
dim_customers = spark.table("workspace.gold.dim_customers").toPandas()

#Customer level features 

In [0]:
customer_features = (
    fact_sales.groupby("customer_key")
    .agg(
        total_spend=("sales_amount", "sum"),
        purchase_count=("order_number", "count"),
        avg_order_value=("sales_amount", "mean"),
        total_quantity=("quantity", "sum"),
        avg_price=("price", "mean"),
    )
    .reset_index()
)

# Join with customer info
customer_features = customer_features.merge(dim_customers, on="customer_key", how="left")


print(f"Customer features: {len(customer_features)} customers")
print(customer_features[["customer_key", "total_spend", "purchase_count", "avg_order_value"]].head())

#Features for clustering 

In [0]:
features = ["total_spend", "purchase_count", "avg_order_value", "total_quantity", "avg_price"]
X = customer_features[features].fillna(0)

#Scale features

In [0]:
scaler = StandardScaler()
X_scaled =  scaler.fit_transform(X)

# Train KMeans with MLflow tracking 

In [0]:
with mlflow.start_run(run_name="kmeans_segmentation_v1"):
    
    n_clusters = 3
    mlflow.log_param("n_clusters", n_clusters)
    mlflow.log_param("features", str(features))
    mlflow.log_param("n_customers", len(X))

    # Train
    kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
    kmeans.fit(X_scaled)

    # Assign segments
    customer_features["segment"] = kmeans.labels_
    customer_features["segment_label"] = customer_features["segment"].map({
        0: "Low Value",
        1: "Mid Value",
        2: "High Value"
    })
    
    # Log metrics
    inertia = kmeans.inertia_
    mlflow.log_metric("inertia", inertia)

    # Log segment distribution
    segment_counts = customer_features["segment_label"].value_counts()
    for label, count in segment_counts.items():
        mlflow.log_metric(f"segment_{label.replace(' ', '_')}_count", count)

    # Log model
    signature = infer_signature(X_scaled, kmeans.labels_)
    mlflow.sklearn.log_model(
        kmeans,
        artifact_path="customer_segmentation_model",
        registered_model_name="customer_segmentation_v3",
        signature=signature
    )
    
    print("Training complete!")
    print(f"Inertia: {inertia:.2f}")
    print(f"\nSegment distribution:")
    print(segment_counts)
    print(f"\nAverage stats per segment:")
    print(customer_features.groupby("segment_label")[features].mean().round(2))